In [14]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import glob, os
import time

In [15]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)

In [16]:
categorical_features = [
    "ram_type", "display_resolution", "matrix_type", "region", "brand"
    "videocard_brand", "videocard", "os", "rom_type", "processor"
]
numeric_features = [
    "timedelta_minutes", "battery_life", "ram_volume", "diagonal", "rom_volume"
]

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing'))
        ]), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='RMSE',
        cat_features=categorical_features,
        early_stopping_rounds=50,
        random_seed=42,
        verbose=0
    ))
])

In [9]:
pipeline.fit()

TypeError: Pipeline.fit() missing 1 required positional argument: 'X'